<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Llama3_fine_tuning_korquad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3를 KorQuad 데이터셋에 맞게 fine-tuning 하기

In [ ]:
!nvidia-smi

## Library Install and Token Setting

In [ ]:
# [Cell 1] Environment Setup
import torch

# 1. Check GPU Status
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU Detected: {gpu_name} (Capability {capability[0]}.{capability[1]})")
else:
    print("No GPU found. Please checking Runtime type.")

print("\nInstalling Unsloth (Colab T4 Optimized)...")

# 2. Install Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Install Dependencies
print("\nInstalling Core Dependencies...")
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes xformers

print("\nSetup Completed.")

## Korquad 데이터셋에 맞게 Llama3 Fine-Tuning
## AutoTrain이 충돌을 일으켜서 unsloth을 사용하였다

In [ ]:
# [Cell 2] 학습 실행
import os
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import login

# --- 기본 설정 ---
hf_token = "Input Your Token"
login(hf_token)

project_name = "llama3-korquad-finetuning-da-8B"
data_path = "korquad_prompt_da"
max_seq_length = 2048 # KorQuAD 지문 길이 고려

print(f"Start Project: {project_name}")

# 1. 모델 로드 (4bit 양자화)
# T4 환경이라 load_in_4bit=True 필수
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# LoRA 설정
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # 0 권장 (속도 이슈)
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 2. 데이터 로드 및 전처리
if os.path.exists(data_path):
    import glob
    files = glob.glob(f"{data_path}/*.json")

    if not files:
        raise FileNotFoundError(f"'{data_path}' 안에 json 파일 없음. 경로 확인 바람.")

    # json 포맷에 따라 read 방식 분기 처리
    df_list = []
    for f in files:
        try:
            df_list.append(pd.read_json(f))
        except ValueError:
            # 일반 json 아니면 jsonl로 시도
            df_list.append(pd.read_json(f, lines=True))

    df = pd.concat(df_list)
    print(f"Raw Data Loaded: {len(df)} rows")

    # 'text' 컬럼 없으면 Llama-3 포맷으로 생성
    if 'text' not in df.columns:
        print("'text' 컬럼 생성 중... (Llama-3 Chat Format)")

        def formatting_prompts_func(row):
            # 컬럼명 대소문자 섞여있을 경우 대비
            ctx = row.get('context', row.get('Context', ''))
            qst = row.get('question', row.get('Question', ''))
            ans = row.get('answer', row.get('Answer', ''))

            # 시스템 프롬프트
            sys_msg = "You are a helpful AI assistant. Answer the question based on the context."

            return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{sys_msg}<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: {ctx}
Question: {qst}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{ans}<|eot_id|>"""

        df['text'] = df.apply(formatting_prompts_func, axis=1)

    dataset = Dataset.from_pandas(df)
    print(f"Ready to Train: {len(dataset)} samples")

else:
    raise FileNotFoundError(f"폴더 없음: {data_path}")


# 3. Trainer 설정
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # 배치 2*4 = 8 효과
        warmup_ratio = 0.1,
        num_train_epochs = 15,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = project_name,
        report_to = "none", # wandb log 끔
    ),
)

# 학습 시작
print("Training Start...")
trainer_stats = trainer.train()

print(f"Done! Saved to '{project_name}'")


# 학습결과 zip 파일로 압축후 다운로드

In [ ]:
import zipfile
import shutil
from google.colab import files

# 압축할 폴더 이름
folder_name = "llama3-korquad-finetuning-da-8B"  # Data Augmentation 적용 o

# 생성될 ZIP 파일 이름
zip_file_name = "llama3-korquad-finetuning-da-8B.zip" # Data Augmentation 적용 o

# 폴더를 ZIP 파일로 압축
shutil.make_archive(zip_file_name[:-4], 'zip', folder_name)

# ZIP 파일을 로컬로 다운로드
files.download(zip_file_name)

##Llama-3-8B 모델 학습 후 성능측정

In [ ]:
# [Cell 3] 모델 저장 및 Inference Test
from unsloth import FastLanguageModel
import torch

# --- Config ---
project_name = "llama3-korquad-finetuning-da-8B"

# 1. Save Model (Adapter)
# 로컬 폴더에 LoRA 어댑터와 토크나이저 저장
print(f"Saving model to '{project_name}'...")
model.save_pretrained(project_name)
tokenizer.save_pretrained(project_name)
print("Save Completed.")

# 2. Inference Setup
# 추론 모드로 전환 (Gradient 계산 꺼서 속도 향상)
FastLanguageModel.for_inference(model)

def apply_chat_template(question, context=""):
    # Llama-3 Chat Prompt Template 적용
    sys_msg = "You are a helpful AI assistant. Answer the question based on the context."

    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{sys_msg}<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: {context}
Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

# 3. Test Inputs
question = "임종석이 1989년 2월 15일에 지명수배된 혐의는 무엇인가?"
context = "1989년 2월 15일 여의도 농민 폭력 시위를 주도한 혐의로 지명수배된 임종석은..."

# 4. Generate
print("\n Generating answer...")

prompt = apply_chat_template(question, context)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=64, # 생성 길이 제한
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id # Warning 방지용
)

# 5. Decode & Parse Output
decoded_output = tokenizer.batch_decode(outputs)[0]

# 프롬프트 부분은 잘라내고, 순수 답변만 추출
response = decoded_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
# 끝에 붙을 수 있는 eot 태그 제거
response = response.replace("<|eot_id|>", "")

print("-" * 50)
print(f"Q: {question}")
print(f"A: {response}")
print("-" * 50)